# Selective cost: 1-hour moves by hour of day and volatility regime (Phase 9, step 1)

Phase 9 of the intraday track starts with 1-hour holding periods (see the decision in notebook 10). This notebook measures whether selective conditions make the typical 1-hour move larger relative to the 0.30% round-trip cost. BTC/USDT only, using the existing 15-minute data; no new data is downloaded. It is descriptive: it defines no setup and makes no trading decision.

**Questions**
1. Typical 1-hour move and cost ratio by the hour of day (UTC) at which the period starts, for 2017–2023 and for 2024 onward.
2. The same, grouped by the previous day's daily volatility regime ("high" / "low").
3. The same for hour of day × previous day's volatility regime, 2024 onward only, with the number of periods in each group.

**Definitions (fixed before any result was computed)**
- 1-hour periods start on the hour (UTC) and follow each other without overlap. The return of a period is log(open at period end / open at period start), using 15-minute candle opens, as in notebook 10.
- A period is dropped if the 15-minute candle at its start or its end is missing.
- Typical move = sample standard deviation of the returns in a group. Cost ratio = 0.30% / typical move.
- Hour of day and year are those of the period start (UTC). 2017–2023 = periods starting before 2024-01-01; 2024 onward = periods starting on or after 2024-01-01.
- Volatility regime: `vol_regime` from `add_regimes(add_features(...))` on daily BTC/USDT. A period starting on day D gets the label of day D − 1 (the previous calendar day), which is final at the close of D − 1. No information from day D is used. Periods whose previous day has no label (the first 384 days of daily data) are excluded from questions 2 and 3 and counted.

## Step 0: Imports and settings

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from cryptosense.data import load_processed, timeframe_to_ms
from cryptosense.features import add_features
from cryptosense.regimes import add_regimes

DATA_DIR = Path.cwd().parent / "data"
SYMBOL = "BTC/USDT"
COST_PER_SIDE = 0.0015
ROUND_TRIP = 2 * COST_PER_SIDE
HOUR_MS = timeframe_to_ms("1h")
SPLIT = pd.Timestamp("2024-01-01", tz="UTC")

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 100)

## Step 1: Load 15-minute and daily data

The existing processed files are loaded; no download. Daily data gets features and regimes.

In [2]:
m15 = load_processed(SYMBOL, DATA_DIR, timeframe="15m")
daily = add_regimes(add_features(load_processed(SYMBOL, DATA_DIR)))

print(f"15m:   {len(m15):,} candles, {m15.index[0]} -> {m15.index[-1]}")
print(f"daily: {len(daily):,} days, {daily.index[0].date()} -> {daily.index[-1].date()}")
labelled = daily["vol_regime"].dropna()
print(f"vol_regime labelled: {labelled.index[0].date()} -> {labelled.index[-1].date()} ({len(labelled):,} days)")

# 15-minute opens looked up by integer candle timestamp (ms, UTC)
open_15m = m15.set_index("timestamp")["open"]

15m:   321,296 candles, 2017-08-17 04:00:00+00:00 -> 2026-10-21 21:00:00+00:00
daily: 3,352 days, 2017-08-17 -> 2026-10-20
vol_regime labelled: 2018-09-05 -> 2026-10-20 (2,968 days)


## Step 2: Non-overlapping 1-hour returns with hour, sample and previous-day regime

Each 1-hour period gets its start hour (UTC), its sample (2017–2023 or 2024 onward) and the volatility regime of the previous calendar day.

In [3]:
first_ts, last_ts = int(open_15m.index[0]), int(open_15m.index[-1])
starts = np.arange(-(-first_ts // HOUR_MS) * HOUR_MS, last_ts - HOUR_MS + 1, HOUR_MS)
p_start = open_15m.reindex(starts).to_numpy()
p_end = open_15m.reindex(starts + HOUR_MS).to_numpy()
start_time = pd.to_datetime(starts, unit="ms", utc=True)

hours = pd.DataFrame({"log_return": np.log(p_end / p_start)}, index=start_time)
print(f"1-hour periods on the grid: {len(hours):,} | dropped (missing candle): {int(hours['log_return'].isna().sum()):,}")
hours = hours.dropna()

hours["hour_utc"] = hours.index.hour
hours["sample"] = np.where(hours.index < SPLIT, "2017–2023", "2024 onward")

# Label of the previous calendar day: final at the close of D - 1, so nothing from day D is used
regime_by_day = daily["vol_regime"]
regime_by_day.index = regime_by_day.index.as_unit("ns")
prev_day = hours.index.normalize() - pd.Timedelta(days=1)
hours["prev_vol_regime"] = regime_by_day.reindex(prev_day.as_unit("ns")).to_numpy()

print(f"1-hour returns used: {len(hours):,}")
print(f"without a previous-day regime label: {int(hours['prev_vol_regime'].isna().sum()):,} "
      f"(starts {hours.index[0]} -> {hours.index[hours['prev_vol_regime'].isna()][-1]})")
print()
print(hours.groupby(["sample", "prev_vol_regime"], dropna=False).size().rename("periods").to_string())

1-hour periods on the grid: 80,465 | dropped (missing candle): 173
1-hour returns used: 80,292
without a previous-day regime label: 9,167 (starts 2017-08-17 04:00:00+00:00 -> 2018-09-05 23:00:00+00:00)

sample       prev_vol_regime
2017–2023    high               20831
             low                25697
             NaN                 9167
2024 onward  high               11688
             low                12909


## Step 3: Statistics helper

For each group: number of 1-hour periods (n), typical move (standard deviation, in %) and cost ratio (0.30% / typical move).

In [4]:
def cost_stats(r):
    """n, typical move (%) and round-trip cost / typical move for one group of returns."""
    move = r.std()
    return pd.Series({"n": len(r), "typical_move_%": 100 * move, "cost_ratio": ROUND_TRIP / move})


def grouped(frame, by):
    """cost_stats for each group of ``frame`` by the column(s) ``by``."""
    table = frame.groupby(by)["log_return"].apply(cost_stats).unstack()
    table["n"] = table["n"].astype(int)
    return table


def show(table):
    print(table.to_string(formatters={
        col: (lambda v: f"{v:,.0f}") if str(col).endswith("n") or col == "n" else (lambda v: f"{v:.3f}")
        for col in table.columns
    }))

## Step 4: By hour of day (UTC), 2017–2023 and 2024 onward

The "all hours" row covers every hour of the sample.

In [5]:
by_hour = {}
for sample, frame in hours.groupby("sample"):
    table = grouped(frame, "hour_utc")
    table.loc["all hours"] = cost_stats(frame["log_return"])
    table["n"] = table["n"].astype(int)
    by_hour[sample] = table

hour_table = pd.concat(by_hour, axis=1)
hour_table.index.name = "hour_utc"
print(hour_table.to_string(formatters={
    col: (lambda v: f"{v:,.0f}") if col[1] == "n" else (lambda v: f"{v:.3f}")
    for col in hour_table.columns
}))

          2017–2023                           2024 onward                          
                  n typical_move_% cost_ratio           n typical_move_% cost_ratio
hour_utc                                                                           
0             2,324          0.929      0.323       1,025          0.494      0.607
1             2,311          0.873      0.343       1,025          0.558      0.538
2             2,310          0.869      0.345       1,025          0.476      0.631
3             2,310          0.758      0.396       1,025          0.403      0.744
4             2,312          0.767      0.391       1,025          0.373      0.805
5             2,314          0.720      0.417       1,025          0.373      0.805
6             2,314          0.756      0.397       1,025          0.440      0.681
7             2,317          0.784      0.382       1,025          0.416      0.720
8             2,319          0.815      0.368       1,025          0.450    

## Step 5: By the previous day's volatility regime, 2017–2023 and 2024 onward

Only periods whose previous day has a regime label.

In [6]:
labelled_hours = hours.dropna(subset=["prev_vol_regime"])
regime_table = grouped(labelled_hours, ["sample", "prev_vol_regime"])
show(regime_table)

                                 n typical_move_% cost_ratio
sample      prev_vol_regime                                 
2017–2023   high            20,831          0.897      0.334
            low             25,697          0.570      0.527
2024 onward high            11,688          0.556      0.540
            low             12,909          0.448      0.669


## Step 6: Hour of day × previous day's volatility regime, 2024 onward

Number of 1-hour periods, typical move (%) and cost ratio for each combination.

In [7]:
recent = labelled_hours[labelled_hours["sample"] == "2024 onward"]
combo = grouped(recent, ["hour_utc", "prev_vol_regime"]).unstack("prev_vol_regime")
combo = combo.swaplevel(axis=1)[["high", "low"]]
combo = combo.reindex(columns=["n", "typical_move_%", "cost_ratio"], level=1)
print(combo.to_string(formatters={
    col: (lambda v: f"{v:,.0f}") if col[1] == "n" else (lambda v: f"{v:.3f}")
    for col in combo.columns
}))

prev_vol_regime high                           low                          
                   n typical_move_% cost_ratio   n typical_move_% cost_ratio
hour_utc                                                                    
0                487          0.567      0.529 538          0.418      0.718
1                487          0.624      0.481 538          0.491      0.611
2                487          0.506      0.593 538          0.446      0.673
3                487          0.417      0.719 538          0.391      0.768
4                487          0.438      0.685 538          0.302      0.994
5                487          0.416      0.720 538          0.329      0.913
6                487          0.511      0.587 538          0.365      0.821
7                487          0.470      0.639 538          0.362      0.830
8                487          0.496      0.605 538          0.405      0.742
9                487          0.407      0.737 538          0.356      0.844

## Results

**Data.** 80,465 1-hour periods on the grid, 173 dropped for a missing endpoint candle, 80,292 used (2017–2023: 55,695; 2024 onward: 24,597). Daily `vol_regime` is labelled from 2018-09-05; 9,167 periods (starting 2017-08-17 04:00 → 2018-09-05 23:00) have no previous-day label and are excluded from the regime tables. Typical move = standard deviation of 1-hour log returns (%); cost ratio = 0.30% / typical move.

### By hour of day (UTC)

n per hour: 2017–2023 2,310–2,327; 2024 onward 1,024–1,025.

| Hour | 2017–2023 move | 2017–2023 ratio | 2024+ move | 2024+ ratio |
|---|---|---|---|---|
| 00 | 0.929 | 0.323 | 0.494 | 0.607 |
| 01 | 0.873 | 0.343 | 0.558 | 0.538 |
| 02 | 0.869 | 0.345 | 0.476 | 0.631 |
| 03 | 0.758 | 0.396 | 0.403 | 0.744 |
| 04 | 0.767 | 0.391 | 0.373 | 0.805 |
| 05 | 0.720 | 0.417 | 0.373 | 0.805 |
| 06 | 0.756 | 0.397 | 0.440 | 0.681 |
| 07 | 0.784 | 0.382 | 0.416 | 0.720 |
| 08 | 0.815 | 0.368 | 0.450 | 0.666 |
| 09 | 0.789 | 0.380 | 0.381 | 0.788 |
| 10 | 0.939 | 0.319 | 0.370 | 0.811 |
| 11 | 0.776 | 0.387 | 0.373 | 0.804 |
| 12 | 0.931 | 0.322 | 0.446 | 0.673 |
| 13 | 0.936 | 0.320 | 0.630 | 0.476 |
| 14 | 1.015 | 0.296 | 0.775 | 0.387 |
| 15 | 0.917 | 0.327 | 0.733 | 0.409 |
| 16 | 0.954 | 0.314 | 0.548 | 0.548 |
| 17 | 0.817 | 0.367 | 0.589 | 0.509 |
| 18 | 0.781 | 0.384 | 0.538 | 0.558 |
| 19 | 0.810 | 0.370 | 0.542 | 0.554 |
| 20 | 0.879 | 0.341 | 0.494 | 0.608 |
| 21 | 0.853 | 0.352 | 0.472 | 0.636 |
| 22 | 0.825 | 0.364 | 0.490 | 0.612 |
| 23 | 0.912 | 0.329 | 0.404 | 0.743 |
| All hours | 0.854 | 0.351 | 0.502 | 0.598 |

### By previous day's volatility regime

| Sample | Regime | n | Move | Ratio |
|---|---|---|---|---|
| 2017–2023 | high | 20,831 | 0.897 | 0.334 |
| 2017–2023 | low | 25,697 | 0.570 | 0.527 |
| 2024 onward | high | 11,688 | 0.556 | 0.540 |
| 2024 onward | low | 12,909 | 0.448 | 0.669 |

### Hour of day × previous day's volatility regime, 2024 onward

| Hour | high n | high move | high ratio | low n | low move | low ratio |
|---|---|---|---|---|---|---|
| 00 | 487 | 0.567 | 0.529 | 538 | 0.418 | 0.718 |
| 01 | 487 | 0.624 | 0.481 | 538 | 0.491 | 0.611 |
| 02 | 487 | 0.506 | 0.593 | 538 | 0.446 | 0.673 |
| 03 | 487 | 0.417 | 0.719 | 538 | 0.391 | 0.768 |
| 04 | 487 | 0.438 | 0.685 | 538 | 0.302 | 0.994 |
| 05 | 487 | 0.416 | 0.720 | 538 | 0.329 | 0.913 |
| 06 | 487 | 0.511 | 0.587 | 538 | 0.365 | 0.821 |
| 07 | 487 | 0.470 | 0.639 | 538 | 0.362 | 0.830 |
| 08 | 487 | 0.496 | 0.605 | 538 | 0.405 | 0.742 |
| 09 | 487 | 0.407 | 0.737 | 538 | 0.356 | 0.844 |
| 10 | 487 | 0.418 | 0.717 | 538 | 0.319 | 0.940 |
| 11 | 487 | 0.410 | 0.731 | 538 | 0.336 | 0.892 |
| 12 | 487 | 0.489 | 0.613 | 538 | 0.403 | 0.745 |
| 13 | 487 | 0.679 | 0.442 | 538 | 0.580 | 0.517 |
| 14 | 487 | 0.866 | 0.346 | 538 | 0.683 | 0.439 |
| 15 | 487 | 0.808 | 0.371 | 538 | 0.658 | 0.456 |
| 16 | 487 | 0.618 | 0.486 | 538 | 0.476 | 0.631 |
| 17 | 487 | 0.612 | 0.490 | 538 | 0.568 | 0.528 |
| 18 | 487 | 0.618 | 0.485 | 538 | 0.452 | 0.664 |
| 19 | 487 | 0.610 | 0.492 | 538 | 0.472 | 0.636 |
| 20 | 487 | 0.553 | 0.543 | 538 | 0.434 | 0.692 |
| 21 | 487 | 0.515 | 0.583 | 537 | 0.430 | 0.698 |
| 22 | 487 | 0.526 | 0.570 | 537 | 0.456 | 0.659 |
| 23 | 487 | 0.448 | 0.670 | 537 | 0.359 | 0.835 |

Lowest cost ratio, 2024 onward: 14:00 (0.387) by hour; 14:00 × high (0.346) by combination. Highest: 10:00 (0.811) by hour; 04:00 × low (0.994) by combination.